# Wykrywanie ukrytego ryzyka w strukturach offshore

## Analiza ICIJ Paradise Papers w Jenner

Ten notatnik uruchamia kompletny, kompleksowy potok analityki
nadużyć na prawdziwym wycieku ICIJ Paradise Papers — **163,435
węzłów** obejmujących 24,957 podmiotów offshore, 77,012 członków
kierownictwa, 59,228 adresów oraz 2,031 pośredników, powiązanych
221,112 relacjami OFFICER_OF.

Źródłem danych jest współdzielona usługa `step-neo4j` platformy
Jenner Workspace — Neo4j 5.26 Community Edition z wtyczką Graph Data
Science, przechowująca publiczny zrzut ICIJ Paradise Papers, w trybie
tylko-do-odczytu na poziomie serwera. Kapsuły (pody) Workspace łączą
się z nią pod adresem `bolt://step-neo4j:7687` za pośrednictwem
zmiennych środowiskowych `JENNER_NEO4J_HOST` oraz `JENNER_NEO4J_PASS`
wbudowanych w każdą kapsułę Workspace przez platformę; żadna
konfiguracja per-najemca nie jest wymagana. Każda komórka tego
notatnika działa na tym żywym grafie — nigdzie w potoku nie ma danych
syntetycznych ani próbkowanych.

Analiza jest zorganizowana w piętnaście sekcji, opakowanych w jedną
dyrektywę `ODS PDF`, tak aby pisemny raport zawierał całą historię:

| Sekcja | Temat |
|---|---|
| 1 | Połączenie i oszacowanie rozmiaru danych |
| 2 | Rozkład jurysdykcji |
| 3 | Wykrywanie społeczności metodą Louvain |
| 4 | Centralność PageRank |
| 5 | Inżynieria cech na poziomie podmiotu |
| 6 | Przesiew względem listy OFAC-SDN |
| 7 | Przeżycie Kaplana-Meiera |
| 8 | Proporcjonalne hazardy Coxa |
| 9 | Regresja logistyczna złożoności |
| 10 | Skonsolidowane statystyki nagłówkowe |
| 11 | Ranking wpływu skupiony na członkach kierownictwa |
| 12 | Wzorce zakładania podmiotów w czasie |
| 13 | Porównanie międzywyciekowe |
| 14 | Szersze wzbogacenie danymi OpenSanctions |
| 15 | Złożony ranking ryzyka podmiotów |

**Podstawowe źródło danych:** International Consortium of
Investigative Journalists, *Paradise Papers* (2017). Publiczny zrzut
dostępny pod adresem
<https://offshoreleaks.icij.org/pages/database>.

**Dane pomocnicze zapisane w `data/`:**

- `data/ofac_sdn.csv` — próbka amerykańskiej listy OFAC Specially
  Designated Nationals (500 wierszy, kwiecień 2026)
- `data/opensanctions_default.csv` — skonsolidowana migawka sankcji
  OpenSanctions, 2026-04-17
- `data/tax_haven_ranks.csv` — opublikowane pozycje z Corporate Tax
  Haven Index 2021 sieci Tax Justice Network

## 1. Połączenie i oszacowanie rozmiaru danych

Otwieramy połączenie `LIBNAME ... GRAPH ENGINE=NEO4J` do
współdzielonego hosta badawczego. Jądro ma ustawione w swoim
środowisku host oraz hasło, więc odczyt SYSGET rozwiązuje się
automatycznie.

In [3]:
/* Otwieramy pojedyncze opakowanie ODS PDF wokół całej analizy. Każdy wynik
   PROC od Sekcji 1 wzwyż jest przechwytywany w tym raporcie. PDF jest
   zamykany na samym końcu notatnika, aby pisemny raport zawierał pełną
   narrację: skalę, jurysdykcje, społeczności, PageRank, cechy, sankcje,
   przeżycie, Coxa, regresję logistyczną, perspektywę członków kierownictwa,
   wymiar czasowy, porównanie międzywyciekowe, szersze sankcje oraz końcowy
   złożony ranking ryzyka. */
ods pdf plik="output/icij_fraud_report.pdf" style=journal;

tytuł "ICIJ Paradise Papers — Hidden-Risk Analysis";

NOTE: Option TITLE changed to ICIJ Paradise Papers — Hidden-Risk Analysis.


In [5]:
/* Ustalamy lokalizację zapisanych plików CSV demonstracji. Domyślnie:
   ścieżka względna `data/` (rozwiązuje się, gdy CWD jądra jest katalogiem
   notatnika — standardowe zachowanie Jupyter). Nadpisanie: ustaw
   JENNER_ICIJ_DATA w środowisku jądra na ścieżkę bezwzględną, jeśli jądro
   jest uruchamiane z innego CWD. */
%let _raw_icij_data = %sysget(JENNER_ICIJ_DATA);
%let _icij_data = %sysfunc(ifc(%długość(%superq(_raw_icij_data))=0,
                              dane, %superq(_raw_icij_data)));
%zapisz NOTE: ICIJ demo dane directory: &_icij_data;

%let _host = %sysget(JENNER_NEO4J_HOST);
%let _pwd  = %sysget(JENNER_NEO4J_PASS);

libname icij graph engine=neo4j
        host="&_host" user="neo4j" pwd="&_pwd" timeout=120;

procedura gql conn=icij out=node_total;
    query "MATCH (n) RETURN count(n) AS total_nodes";
quit;

procedura gql conn=icij out=label_counts;
    query "
        MATCH (e:Entity)       WITH count(e) AS n_entity
        MATCH (o:Officer)      WITH n_entity, count(o) AS n_officer
        MATCH (i:Intermediary) WITH n_entity, n_officer,
                                     count(i) AS n_intermediary
        MATCH (a:Address)      WITH n_entity, n_officer, n_intermediary,
                                     count(a) AS n_address
        RETURN n_entity, n_officer, n_intermediary, n_address
    ";
quit;

/* Pokazujemy liczności za pomocą PROC MEANS SUM (każdy zbiór danych to
   jednowierszowa liczność, więc SUM == wartość — daje klasyczne pole
   podsumowania SAS bez sztuczki DATA _null_ PUT). */
procedura średnie dane=node_total sum maxdec=0;
    zmienna total_nodes;
    tytuł "Total nodes in the Paradise-Papers leaked graph";
wykonaj;

procedura średnie dane=label_counts sum maxdec=0;
    zmienna n_entity n_officer n_intermediary n_address;
    etykieta n_entity       = "Entities"
          n_officer      = "Officers"
          n_intermediary = "Intermediaries"
          n_address      = "Addresses";
    tytuł "Node counts by label";
wykonaj;

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable              Sum
 --------------------------
 total_nodes         163414
 --------------------------

                  ICIJ Paradise Papers — Hidden-Risk Analysis                   

                  ICIJ Paradise Papers — Hidden-Risk Analysis                  1

                              The MEANS Procedure

 Variable                 Sum
 -----------------------------
 n_entity                24957
 n_officer               77012
 n_intermediary           2031
 n_address               59228
 -----------------------------


NOTE: Graph library ICIJ assigned engine=NEO4J (auth=STATIC).
NOTE: PROC GQL conn=icij mode=Read

NOTE: PROC GQL: wrote 1 observation and 1 variable to node_total.

NOTE: Wrote node_total (1 rows, 1 columns).
NOTE: PROC GQL elapsed:
  wall 

## 2. Gdzie rejestrowany jest kapitał?

Wyciek Paradise Papers obejmuje podmioty zarejestrowane w około 50
jurysdykcjach. Poziomy wykres słupkowy dla dziesięciu największych
jurysdykcji pokazuje, jak bardzo aktywność offshore koncentruje się
w garstce terytoriów uprzywilejowanych podatkowo. Dominują Bermudy
i Kajmany — łącznie 18,204 podmiotów (73% z 24,957 nazwanych).

In [ ]:
procedura gql conn=icij out=jurisdictions;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        RETURN e.jurisdiction            AS jurisdiction,
               e.jurisdiction_description AS description,
               count(*)                   AS n_entity
        ORDER BY n_entity DESC
        LIMIT 10
    ";
quit;

procedura drukuj dane=jurisdictions etykieta;
    tytuł "Top 10 Paradise-Papers Jurisdictions";
    etykieta jurisdiction="Jurisdiction" description="Description" n_entity="Entities";
    format n_entity comma12.;
wykonaj;

/* Przygotowanie Pareto: zapytanie Cypher już porządkuje jurysdykcje od
   największej do najmniejszej, więc akumulujemy sumę bieżącą i wyrażamy ją
   jako procent sumy z pierwszej dziesiątki. Nakładka liniowa na osi
   pomocniczej wznosi się od pierwszej jurysdykcji do 100% przy dziesiątej. */
procedura sql noprint;
    wybierz sum(n_entity) into :_grand
    from jurisdictions;
quit;

dane jurisdictions_pareto;
    ustaw jurisdictions;
    przechowaj _cum 0;
    _cum + n_entity;
    cum_pct = 100 * _cum / &_grand;
    usuń _cum;
wykonaj;

procedura sgplot dane=jurisdictions_pareto;
    vbar jurisdiction / response=n_entity
                        categoryorder=respdesc
                        fillattrs=(color=steelblue);
    vline jurisdiction / response=cum_pct stat=sum y2axis
                         lineattrs=(color=darkred thickness=2);
    xaxis etykieta="Jurisdiction (ISO-2)";
    yaxis etykieta="Number of Entities";
    y2axis etykieta="Cumulative % of top-10 total" min=0 max=100
           values=(0 20 40 60 80 100);
    tytuł "Top 10 Paradise-Papers Jurisdictions — Pareto";
wykonaj;
tytuł;

## 3. Kto tworzy skupiska? Wykrywanie społeczności metodą Louvain

`PROC NETWORK` uruchamia algorytm Louvain lokalnie na dwudzielnym
grafie `(Officer)-[OFFICER_OF]->(Entity)`, pobierając za pomocą
zapytania Cypher `MATCH` (tylko do odczytu) do `step-neo4j` podgraf
pięciu tysięcy członków kierownictwa o najwyższym stopniu.
Współdzielony `step-neo4j` platformy wymusza
`server.databases.default_to_read_only=true`, więc każda analityka
grafowa, która modyfikowałaby bazę danych (krok GDS
`gds.graph.project`, który wywołałby `PROC LINKS`), jest blokowana na
serwerze. `PROC NETWORK` omija to ograniczenie — przesyła dopasowane
wiersze przez Bolt i uruchamia algorytm w procesie, wewnątrz kapsuły
Workspace.

Próbkujemy do 5,000 najlepiej połączonych członków kierownictwa,
ponieważ Louvain na pełnym grafie dwudzielnym (165 tys. krawędzi)
zajmuje w NetworkX kilka minut, podczas gdy Java GDS robi to w
sekundy; dla interaktywnego tempa demonstracji podgraf zachowuje
analityczny przekaz (struktura społeczności pośredników o dużym
wolumenie) i utrzymuje krótki czas działania.

Następnie dołączamy etykiety społeczności z powrotem do tabeli
podmiotów, aby dalsze sekcje mogły oszacować rozmiar strukturalnych
skupisk.

In [ ]:
procedura network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = community_nodes;
    linksvar from=a_node_id to=b_node_id;
    community algorithm=louvain;
wykonaj;

/* Zmieniamy nazwę kolumny `community_1` z PROC NETWORK na nazwę
   `community_id`, której oczekuje dalsza PROC FREQ. */
dane community;
    ustaw community_nodes(zachowaj=node community_1
                        przemianuj=(community_1=community_id));
wykonaj;

procedura częstości dane=community order=częstości;
    tables community_id / noprint out=community_sizes;
wykonaj;

dane top_comms;
    ustaw community_sizes;
    gdzie COUNT >= 200;
    zachowaj community_id COUNT;
    przemianuj COUNT = community_size;
wykonaj;

In [ ]:

procedura drukuj dane=top_comms (obs=15) etykieta;
    tytuł "Largest Louvain communities — node counts";
    format community_id community_size comma12.;
    etykieta community_id="Community ID" community_size="Community Size";
wykonaj;

## 4. Kim są centralni aktorzy? Centralność wektora własnego

Centralność wektora własnego, obliczana w procesie przez
`PROC NETWORK` na tym samym grafie dwudzielnym, wskazuje członków
kierownictwa, których powiązania same łączą się z węzłami o wysokim
stopniu. Jest to najbliższy wewnętrzny odpowiednik PageRank w
warunkach ograniczenia bazy do trybu tylko-do-odczytu, a względne
uporządkowanie członków kierownictwa o wysokiej centralności
odpowiada temu, co wcześniej wytworzył GDS PageRank.

In [ ]:
procedura network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id, top.name AS a_name,
                        e.node_id   AS b_node_id, e.name   AS b_name"
    direction = undirected
    outNodes  = pagerank_nodes;
    linksvar from=a_node_id to=b_node_id;
    centrality eigen=unweight;
wykonaj;

/* Centralność wektora własnego jest najbliższym wewnętrznym odpowiednikiem
   PageRank dla nieskierowanego grafu dwudzielnego; względne uporządkowanie
   członków kierownictwa o wysokiej centralności jest zgodne z tym, co
   wcześniej wytworzył GDS PageRank. Dalszy złożony wskaźnik z Sekcji 11
   łączy po `node_id`, więc zmieniamy nazwę kolumny `node` z PROC NETWORK. */
dane pagerank;
    ustaw pagerank_nodes(zachowaj=node centr_eigen_unwt
                       przemianuj=(node=node_id
                               centr_eigen_unwt=score));
wykonaj;

procedura sortuj dane=pagerank out=pr_sorted;
    według malejąco score;
wykonaj;

dane pr_top;
    ustaw pr_sorted (obs=20);
wykonaj;

procedura drukuj dane=pr_top;
    tytuł "Top 20 PageRank-equivalent (eigenvector centrality) nodes";
wykonaj;

## 5. Zbiór cech na poziomie podmiotu

Do modelowania statystycznego potrzebujemy płaskiej tabeli cech na
poziomie podmiotu. To zapytanie pobiera jurysdykcję, daty
zarejestrowania i zamknięcia, dostawcę usług oraz stopień podgrafu
członków kierownictwa/pośredników każdego podmiotu. Wynikiem jest
zbiór danych liczący 24,957 wierszy — pełna populacja nazwanych
podmiotów Paradise Papers.

In [ ]:
procedura gql conn=icij out=entity_features;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (officer:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(officer) AS officer_count
        OPTIONAL MATCH (src)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, count(src) AS intermediary_count
        RETURN e.node_id           AS node_id,
               e.name              AS entity_name,
               e.jurisdiction      AS jurisdiction,
               e.incorporation_date AS incorporation_date,
               e.closed_date       AS closed_date,
               e.sourceID          AS source_id,
               officer_count,
               intermediary_count
    ";
quit;

procedura średnie dane=entity_features n mean std min p25 median p75 max;
    zmienna officer_count intermediary_count;
    tytuł "Per-entity officer and intermediary counts";
wykonaj;

/* Podkorpus Paradise Papers w tym zrzucie to ~99,98% Appleby — podział
   według service_provider byłby trywialnie zdegenerowany. Zamiast tego
   używamy sourceID (kilka źródeł wycieku) jako osi międzykorpusowej w sekcji
   13. */


## 6. Przesiew względem sankcji OFAC

Przesiewamy zarówno nazwiska członków kierownictwa, jak i nazwy
podmiotów względem amerykańskiej listy Specially Designated Nationals
(SDN) prowadzonej przez Office of Foreign Assets Control (OFAC).
Plik `data/ofac_sdn.csv` w tej demonstracji zawiera 500 prawdziwych
wpisów SDN wylosowanych z pięciu najczęściej używanych programów
(Russia EO 14024, SDGT, SDNTK, GLOMAG, Iran EO 13902) z żywego
publicznego eksportu Departamentu Skarbu pod adresem
`sanctionslistservice.ofac.treas.gov`.

Zastosowana poniżej strategia przesiewu jest **dwuetapowa** i często
używana przez rzeczywiste zespoły compliance:

1. **Dokładne dopasowanie po UPCASE** — najsilniejszy dowód (zwykle
   bezpośrednie trafienie). Dla Paradise Papers zwraca to zero,
   ponieważ zakres danych kończy się w 2014 roku, a większość
   obecnych programów OFAC (takich jak RUSSIA-EO14024 z lutego 2022)
   jest od niego późniejsza.
2. **Dopasowanie po frazie tokenowej (CONTAINS)** — wyodrębnione
   wielowyrazowe frazy z każdej nazwy SDN (usunięte słowa
   nieznaczące, minimalna długość 12 znaków, co najmniej dwa
   znaczące tokeny) używane jako sondy podłańcuchowe. Wychwytuje to
   podmioty, które *dzielą charakterystyczną frazę* z nazwą objętą
   sankcjami.

Lista fraz jest generowana raz z `data/ofac_sdn.csv` i przechowywana
w `ofac_phrases.sas`. Typowy wynik: zero trafień dla członków
kierownictwa i jedno trafienie dla podmiotu — rzeczywiste ustalenie
compliance, że korpus Paradise Papers zawiera niemal żadnych aktorów
obecnie objętych sankcjami z nazwy.

In [ ]:
/* Lista fraz OFAC jest długa — odczytujemy ją z pliku towarzyszącego i
   wstawiamy w miejscu. W uruchomieniu wsadowym (jenner script.jenner) można
   użyć %include; w jądrze Jupyter wstawianie w miejscu jest bardziej
   niezawodne. */
/* Wygenerowane automatycznie z data/ofac_sdn.csv. */
%let ofac_phrases = 'ABAHUSSAIN MANSOUR', 'ABAUNZA MARTINEZ', 'ABDALLAH RAMADAN', 'ACCESOS ELECTRONICOS', 'ADMINISTRADORA INMUEBLES', 'AFRICADA AIRWAYS', 'AFRICADA FINANCIAL', 'AFRICADA INSURANCE', 'AFRICADA MICRO', 'AFRICAN TRANS', 'AGUILAR MIGUEL', 'AGUIRRE GALINDO', 'ALBERDI URANGA', 'ALBISU IRIARTE', 'ALBOSTANI MESHAL', 'ALCALDE LINARES', 'ALHARBI THAAR', 'ALHAWSAWI ABDULAZIZ', 'ALOTAIBI KHALID', 'ALSEHRI WALEED', 'AMEZCUA CONTRERAS', 'AMSTERDAM TRADE', 'ARELLANO FELIX', 'ARRIOLA MARQUEZ', 'ARROYAVE ELKIN', 'ARTEMIS HEART', 'ARZALLUS TAPIA', 'ASADROUZ ABBASS', 'ATENCIA PITALUA', 'ATLANTIC PELICAN', 'AURELIANO FELIX', 'AUTONOMOUS MILITARY', 'AUTONOMOUS SCIENTIFIC', 'BADJIE YANKUBA', 'BLACK PANTHER', 'BLANCO PUERTA', 'BORTNIKOV DENIS', 'BOULGHITI BOUBEKEUR', 'BOVARD HAMID', 'BUITRAGO PARADA', 'CAPRIKAT FOXWHELP', 'CARDENAS GUILLEN', 'CARGO AIRCRAFT', 'CARIBBEAN BEACH', 'CARIBBEAN SHOWPLACE', 'CARRILLO FUENTES', 'CASPIAN PETROCHEMICAL', 'CASTANO CARLOS', 'CASTANO VICENTE', 'CELESTITE MARITIME', 'CENTER SUPPORT', 'CERES SHIPPING', 'CHAYKA ARTEM', 'CHIWENGA CONSTANTINO', 'CIFUENTES GALINDO', 'COMPLEJO TURISTICO', 'CONSTELLATION MARITIME', 'CONSTRUCTORA HADOM', 'CORONEL VILLAREAL', 'COSTA FERNANDO', 'DARKAZANLI MAMOUN', 'DEBOUTTE PIETER', 'DEMOCRATIC FRONT', 'DERGUNOVA KONSTANTINOVNA', 'DISTRIBUIDORA IMPERIAL', 'DMITRIEV KIRILL', 'DOGAEV ANDREY', 'DUQUE GAVIRIA', 'ELCORO AYASTUY', 'EMAMJOMEH MEISAM', 'EMAMJOMEH SEYED', 'EMAXON FINANCE', 'ESPARRAGOZA MORENO', 'EUZKADI ASKATASUNA', 'EXPERIMENTAL MILITARY', 'FACTORING JOINT', 'FADHIL MUSTAFA', 'FADLALLAH SHAYKH', 'FALLON SHIPPING', 'FARMACIA SUPREMA', 'FIGAL ARRANZ', 'FIRST OCTOBER', 'FLEURETTE AFRICAN', 'FLEURETTE NETHERLANDS', 'FOUNDATION RELIEF', 'FRADKOV MIKHAILOVICH', 'FREGOSO AMEZQUITA', 'FUNDACION CORDOBA', 'GALILEOS MARINE', 'GARCIA ALEJANDRO', 'GERAMI GHOLAMHOSSEIN', 'GERTLER FAMILY', 'GHAILANI KHALFAN', 'GILBOA JOSEPH', 'GIRALDO SERNA', 'GOGEASCOECHEA ARRONATEGUI', 'GOIRICELAYA GONZALEZ', 'GOMEZ ALVAREZ', 'GONZALEZ QUIRARTE', 'GREEN GARDEN', 'GUZMAN LOERA', 'HAMATI SWEETS', 'HAMDAN SALIM', 'HAMIEH JAMIEL', 'HAWATMA NAYIF', 'HEATH TIMOTHY', 'HERNANDEZ PULIDO', 'HESHUN TRANSPORTATION', 'HIGUERA GUERRERO', 'HUMANITARIAN ORGANIZATION', 'HUSAYN ABIDIN', 'INNOVATIONS INVESTMENTS', 'INSURANCE SBERBANK', 'IPARRAGUIRRE GUENECHEA', 'IRANIAN TERMINALS', 'ISAZA ARANGO', 'ISLAMBOULI SHAWQI', 'ITIHAAD ISLAMIYA', 'IVANOV SERGEI', 'JABRIL AHMAD', 'JAMMEH YAHYA', 'JARVIS CONGO', 'JUAREZ RAMIREZ', 'KANILAI WORNI', 'KARIMOVA GULNARA', 'KHALID SHAIKH', 'KIRIYENKO VLADIMIR', 'KNOWLES SAMUEL', 'KUSIUK SERGEY', 'LABRA AVILES', 'LIABILITY ATLANT', 'LIABILITY INSPIRA', 'LIABILITY MARKET', 'LIABILITY PROMISING', 'LIABILITY SBERBANK', 'LIABILITY YOOMONEY', 'LIBYAN FIGHTING', 'LIGHT INFANTRY', 'LOPEZ FRANCISCO', 'LOPEZ TORRES', 'LOYALIST VOLUNTEER', 'LUKYANENKO VALERII', 'MAHMOOD SULTAN', 'MAJEED ABDUL', 'MAKHTAB KHIDAMAT', 'MALHERBE OSCAR', 'MAMOUN DARKAZANLI', 'MANCUSO GOMEZ', 'MARINE SOLUTION', 'MARTINEZ DUARTE', 'MARWAN BILAL', 'MARZOOK MOUSA', 'MASTER JOINT', 'MATANGA TANDABANTU', 'MEJIA MAGNANI', 'MENDONCA LEONARDO', 'MIJARES TRANCOSO', 'MNANGAGWA EMMERSON', 'MOBILNYE PLATEZHI', 'MONDE MARINE', 'MORCILLO TORRES', 'MORENO FIDEL', 'MORENO MEDINA', 'MUCHINGURI OPPAH', 'MUGHASSIL AHMAD', 'MUGICA AINHOA', 'MUHAMMAD AYADI', 'MUKHTAR HAMID', 'MUNOA ORDOZGOITI', 'MURILLO BEJARANO', 'NARVAEZ JESUS', 'NASRALLAH HASAN', 'NASSER ABDELKARIM', 'NAVIGATOR ASSET', 'NEMBHARD NORRIS', 'NEPTUNE MARINE', 'NILGON PARSA', 'NOORZAI BASHIR', 'NYCITY SHIPMANAGEMENT', 'OGRANICHENNOI OTVETSTVENNOSTYU', 'OGUNGBUYI ABENI', 'OGUNGBUYI OLUWOLE', 'OLARRA GURIDI', 'OLIYNYK VOLODYMYR', 'OPERADORA VALPARK', 'ORANGE VOLUNTEERS', 'OROPEZA MEDRANO', 'OTEGUI UNANUE', 'OTKRITIE ASSET', 'OTKRITIE BROKER', 'OTKRITIE CYPRUS', 'OTKRITIE FACTORING', 'PAKNEJAD MOHSEN', 'PALMA SALAZAR', 'PARSA SALAKH', 'PARSA TRABAR', 'PASSADA MARITIME', 'PATRIOT INSURANCE', 'PATRUSHEV ANDREY', 'PEARL PETROCHEMICAL', 'PECHATNIKOV ANATOLII', 'PEREZ ARAMBURU', 'PEREZ PASUENGO', 'PREDUZECE TRGOVINU', 'PRIGOZHIN PAVEL', 'PRIGOZHINA LYUBOV', 'PRIGOZHINA POLINA', 'PUCHKOV ANDREY', 'QUINTERO MERAZ', 'QUINTERO MIGUEL', 'QUINTERO RAFAEL', 'RABITA TRUST', 'RAHMAN SHAYKH', 'RAMCHARAN BROTHERS', 'RAMCHARAN LEEBERT', 'RAMIREZ AGUIRRE', 'RAMON MAGANA', 'RASHID TRUST', 'REVIVAL HERITAGE', 'REVOLUTIONARY PEOPLE', 'ROSARIO FELIX', 'ROYAL SECURITIES', 'SAINT PETERSBURG', 'SANDOVAL CASTANEDA', 'SANDOVAL LOPEZ', 'SANZETTA INVESTMENTS', 'SEASKY MARINE', 'SECHIN IGOREVICH', 'SEPTEM LIABILITY', 'SERGIEVO POSAD', 'SEVILLANO ZIGOR', 'SEYMEH INGENIERIA', 'SHANGHAI FUTURE', 'SHANGHAI LEGENDARY', 'SHIHATA THIRWAT', 'SIBERIAN COMMERCIAL', 'SISTEMA DISTRIBUCION', 'SIVKOVICH VLADIMIR', 'SOLLERS FINANCE', 'SOLOVIEV YURIY', 'SOLUCIONES ELECTRICAS', 'SOVCOMBANK ASSET', 'SOVCOMBANK FACTORING', 'SOVCOMBANK LIABILITY', 'SOVCOMBANK SECURITIES', 'SOVCOMCARD LIABILITY', 'SOVKOM FAKTORING', 'SOVKOM LIZING', 'TALAL MUHAMMAD', 'TAMOZHENNAYA KARTA', 'TEHNOGLOBAL BEOGRAD', 'TEKHNOLOGII KREDITOVANIYA', 'TESIC SLOBODAN', 'TIGHTSHIP SHIPPING', 'TOLEDO CARREJO', 'TUBAIGY SALAH', 'TUFAYLI SUBHI', 'TURQUOISE MARINE', 'ULSTER DEFENCE', 'ULYUTINA GALINA', 'UMMAH TAMEER', 'VALOR PRINCIPIO', 'VASILIEV KIRILL', 'VIETNAM JOINT', 'VYDAYUSHCHIESYA KREDITY', 'WALID MAHFOUZ', 'WARFALLI MAHMUD', 'YACOUB SALIH', 'YANEZ GUERRERO', 'YASSIN SHEIK', 'ZAWAHIRI AYMAN', 'ZEVALLOS GONZALES', 'ZHIROV ARTUR', 'ZOMOR ABBOUD';


/* Wielosygnałowe dopasowanie rozmyte względem listy fraz OFAC SDN. 1.
   SOUNDEX — dopasowanie fonetyczne (Russell). Wychwytuje "Zeebell" ~ "Zybl".
   2. SPEDIS — odległość pisowni (≤25 ≈ bliskie dopasowanie). Uwaga: SPEDIS w
   Jenner obecnie używa heurystyki o jednolitym koszcie zamiast ważonego
   modelu kosztów SAS; próg dostrojony pod to. Zgodny z SAS refaktor jest
   śledzony osobno. 3. COMPGED — uogólniona odległość edycyjna × 100 (≤250 =
   do ~2 edycji). Obowiązuje to samo zastrzeżenie Jenner: obecna
   implementacja to Levenshtein × 100, a nie ważone koszty COMPGED z SAS.
   Trafienie z któregokolwiek z trzech sygnałów liczy się jako dopasowanie
   rozmyte. Pobieramy kandydujące nazwiska członków kierownictwa/nazwy
   podmiotów z żywego grafu pojedynczym PROC GQL na każdy rodzaj, a następnie
   uruchamiamy potrójny sygnał w kroku DATA. */

procedura gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer) WHERE o.name IS NOT NULL RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

procedura gql conn=icij out=all_entity_names;
    query "MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.node_id AS node_id, e.name AS entity_name";
quit;

/* Materializujemy listę fraz jako zbiór danych, aby ułatwić łączenie w kroku
   DATA. */
dane ofac_phrase_list;
    długość phrase $80;
    wejście phrase $80.;
    datalines;
;
wykonaj;

/* Wstawiamy te same frazy do zbioru danych — makro powyżej nazywa je na
   potrzeby odniesień w Cypher, ale potrzebujemy także zbioru danych po
   stronie SAS. */
dane ofac_phrase_list;
    długość phrase $80;
    tablica ph [*] $80 _temporary_ (&ofac_phrases);
    powtórz i = 1 to dim(ph);
        phrase = ph[i];
        wyjście;
    koniec;
    usuń i;
wykonaj;

/* Potrójny sygnał rozmyty dla członków kierownictwa. Złączenie krzyżowe +
   wczesne przycinanie po soundex. */
dane officer_hits;
    ustaw all_officer_names;
    długość phrase $80 match_type $10;
    długość on_sx $4 ph_sx $4;
    on_up = upcase(officer_name);
    on_sx = soundex(on_up);
    powtórz k = 1 to n_phrases;
        ustaw ofac_phrase_list (przemianuj=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        jeśli on_sx = ph_sx and on_sx ne '' wtedy powtórz;
            phrase = phrase_k; match_type = 'soundex'; wyjście;
        koniec;
        przeciwnie jeśli spedis(on_up, ph_up) <= 25 wtedy powtórz;
            phrase = phrase_k; match_type = 'spedis';  wyjście;
        koniec;
        przeciwnie jeśli compged(on_up, ph_up) <= 250 wtedy powtórz;
            phrase = phrase_k; match_type = 'compged'; wyjście;
        koniec;
    koniec;
    zachowaj node_id officer_name phrase match_type;
wykonaj;

/* Potrójny sygnał rozmyty dla podmiotów (ten sam kształt). */
dane entity_hits;
    ustaw all_entity_names;
    długość phrase $80 match_type $10;
    długość en_sx $4 ph_sx $4;
    en_up = upcase(entity_name);
    en_sx = soundex(en_up);
    powtórz k = 1 to n_phrases;
        ustaw ofac_phrase_list (przemianuj=(phrase=phrase_k)) point=k nobs=n_phrases;
        ph_up = upcase(phrase_k);
        ph_sx = soundex(ph_up);
        jeśli en_sx = ph_sx and en_sx ne '' wtedy powtórz;
            phrase = phrase_k; match_type = 'soundex'; wyjście;
        koniec;
        przeciwnie jeśli spedis(en_up, ph_up) <= 25 wtedy powtórz;
            phrase = phrase_k; match_type = 'spedis';  wyjście;
        koniec;
        przeciwnie jeśli compged(en_up, ph_up) <= 250 wtedy powtórz;
            phrase = phrase_k; match_type = 'compged'; wyjście;
        koniec;
    koniec;
    zachowaj node_id entity_name phrase match_type;
wykonaj;

procedura częstości dane=officer_hits;
    tables match_type / brakujące;
    tytuł "Officer fuzzy-match breakdown by signal";
wykonaj;

procedura częstości dane=entity_hits;
    tables match_type / brakujące;
    tytuł "Entity fuzzy-match breakdown by signal";
wykonaj;

procedura drukuj dane=officer_hits (obs=25);
    tytuł "First 25 officer fuzzy matches";
wykonaj;

procedura drukuj dane=entity_hits (obs=25);
    tytuł "First 25 entity fuzzy matches";
wykonaj;


## 7. Jak długo żyją podmioty offshore? Kaplan-Meier

12,378 podmiotów Paradise Papers ma zarówno datę zarejestrowania, jak
i datę zamknięcia. Parsowanie osobliwego formatu daty `'2003-Dec-09'`
odbywa się po stronie serwera w Cypher, przy użyciu mapy odwzorowania
kodów miesięcy oraz `duration.inDays`. Wiersze z zastępczą datą
`1900-Jan-01` są wykluczane (reprezentują podmioty, których
rzeczywista data zarejestrowania nie była znana zespołowi badawczemu
ICIJ).

`PROC LIFETEST` stratyfikuje według pięciopoziomowej zmiennej
jurysdykcji (BM, KY, VG, IM, JE oraz koszyk OTHER). Test log-rank
pokazuje, że długości życia podmiotów różnią się dramatycznie między
jurysdykcjami — przy czym podmioty z Wyspy Man przeżywają średnio
około dwa razy dłużej niż podmioty z Bermudów.

In [ ]:
/* Pobieramy pełną tabelę przeżycia jeden raz. Pełny zbiór danych = 12,130
   wierszy. */
procedura gql conn=icij out=surv_raw;
    query "
        WITH {Jan:'01',Feb:'02',Mar:'03',Apr:'04',May:'05',Jun:'06',
              Jul:'07',Aug:'08',Sep:'09',Oct:'10',Nov:'11',Dec:'12'} AS mm
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.closed_date        IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH e, mm,
             split(e.incorporation_date, '-') AS ip,
             split(e.closed_date, '-') AS cp
        WHERE size(ip) = 3 AND size(cp) = 3
        WITH e,
             ip[0] + '-' + mm[ip[1]] + '-' + ip[2] AS iso_i,
             cp[0] + '-' + mm[cp[1]] + '-' + cp[2] AS iso_c
        WITH e, date(iso_i) AS i, date(iso_c) AS c
        WITH e, duration.inDays(i, c).days AS lifespan
        WHERE lifespan > 0
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, lifespan, count(o) AS officer_count
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               lifespan       AS duration,
               officer_count
    ";
quit;

dane surv;
    ustaw surv_raw;
    event = 1;                 /* wszystkie zaobserwowane zamknięcia */
    długość top5 $5;
    top5 = 'OTHER';
    jeśli jurisdiction = 'BM' wtedy top5 = 'BM';
    przeciwnie jeśli jurisdiction = 'KY' wtedy top5 = 'KY';
    przeciwnie jeśli jurisdiction = 'VG' wtedy top5 = 'VG';
    przeciwnie jeśli jurisdiction = 'IM' wtedy top5 = 'IM';
    przeciwnie jeśli jurisdiction = 'JE' wtedy top5 = 'JE';
    log_officers = log(max(1, officer_count));
wykonaj;

procedura częstości dane=surv;
    tables top5 / out=top5_counts;
    tytuł "Entities per jurisdiction group (survival set)";
wykonaj;

`PROC LIFETEST` wyznacza estymator Kaplana-Meiera z kosztem O(n²)
względem rozmiaru warstwy. Aby notatnik pozostał sprawny,
dopasowujemy go do próbki 2,000 wierszy — przebieg trwa około 20
sekund — i pokazujemy wynikowy test log-rank dla różnic między
jurysdykcjami. Model Coxa w następnej sekcji korzysta z tej samej
próbki 2,000 wierszy.

In [ ]:
procedura surveyselect dane=surv out=surv_sample
                   method=srs sampsize=2000 seed=42;
wykonaj;

procedura lifetest dane=surv_sample plots=survival;
    time duration*event(0);
    strata top5;
    tytuł "Kaplan-Meier — entity lifespan by jurisdiction (n=2000)";
wykonaj;

## 8. Hazard zamknięcia — proporcjonalne hazardy Coxa

`PROC PHREG` modeluje hazard zamknięcia jako funkcję jurysdykcji oraz
logarytmu liczby członków kierownictwa. Oszacowania współczynników
hazardu odpowiadają na pytanie compliance: *przy niezmienionych
pozostałych warunkach, o ile szybciej lub wolniej podmiot ulega
zamknięciu w jednej jurysdykcji w porównaniu z inną?*

Oczekiwane ustalenia z rzeczywistych danych:

- Podmioty z Wyspy Man mają około 46% hazardu zamknięcia Bermudów —
  dramatycznie dłuższe życie operacyjne
- Podmioty z Jersey mają około 38% hazardu Bermudów
- Podmioty z Kajmanów mają około 61% hazardu
- Podmioty z Brytyjskich Wysp Dziewiczych niemal dokładnie
  odpowiadają Bermudom
- Każda dodatkowa jednostka logarytmiczna liczby członków
  kierownictwa zmniejsza hazard zamknięcia o mniej więcej 12% —
  większe podmioty utrzymują się dłużej

Wszystkie efekty są wysoce istotne (p < 0.0001 w testach Walda).

In [ ]:
procedura phreg dane=surv_sample;
    klasa top5 / ref=first;
    model duration*event(0) = top5 log_officers;
    tytuł "Cox PH — closure hazard by jurisdiction + log(officers)";
wykonaj;

## 9. Przewidywanie podmiotów o wysokiej złożoności — regresja logistyczna

Definiujemy podmiot **o wysokiej złożoności** jako taki, który ma
pięciu lub więcej członków kierownictwa — mniej więcej górny kwartyl
rozkładu — jako przybliżenie tego rodzaju gęsto obsadzonych struktur,
na których zespoły compliance skupiają się w pierwszej kolejności.
`PROC LOGISTIC` modeluje `high_complexity` jako funkcję jurysdykcji
oraz liczby pośredników.

Wytyczne nakazują próbkowanie z `PLOTS=NONE` do maksymalnie 5,000
wierszy, ponieważ domyślny wykres ROC w `PROC LOGISTIC` ma zachowanie
O(n²) przy dużej skali. Próbkujemy 5,000 podmiotów i używamy
`PLOTS=NONE`.

In [ ]:
procedura surveyselect dane=entity_features out=ef_sample
                   method=srs sampsize=5000 seed=42;
wykonaj;

dane logi;
    ustaw ef_sample;
    długość top5 $5;
    top5 = 'OTHER';
    jeśli jurisdiction = 'BM' wtedy top5 = 'BM';
    przeciwnie jeśli jurisdiction = 'KY' wtedy top5 = 'KY';
    przeciwnie jeśli jurisdiction = 'VG' wtedy top5 = 'VG';
    przeciwnie jeśli jurisdiction = 'IM' wtedy top5 = 'IM';
    przeciwnie jeśli jurisdiction = 'JE' wtedy top5 = 'JE';
    jeśli officer_count >= 5 wtedy high_complexity = 1;
    przeciwnie high_complexity = 0;
wykonaj;

procedura częstości dane=logi;
    tables high_complexity * top5 / norow nocol nopercent;
    tytuł "High-complexity entity rates by jurisdiction";
wykonaj;

procedura logistic dane=logi malejąco plots=none;
    klasa top5;
    model high_complexity = top5 intermediary_count;
    tytuł "Logistic: Pr(>=5 officers) ~ jurisdiction + intermediaries";
wykonaj;

## 10. Skonsolidowane statystyki nagłówkowe

Przerywamy analityczną narrację, aby uchwycić zwięzłe podsumowanie
`PROC MEANS` oraz `PROC FREQ` dla zbioru danych analizy przeżycia.
To jest rodzaj tabeli podsumowującej, od której analityk compliance
lub regulator zaczyna raport. Kolejne sekcje rozszerzają analizę o
ryzyko skupione na członkach kierownictwa, wzorce czasowe,
koncentrację międzywyciekową, szerszy przesiew sankcyjny oraz
końcowy złożony ranking podmiotów. Cały wynik jest przechwytywany w
jednym `ODS PDF` otwartym na początku notatnika i zamkniętym po
Sekcji 15.

In [ ]:
tytuł "ICIJ Paradise Papers — Headline Statistics";

procedura średnie dane=surv n mean std median p25 p75;
    zmienna duration officer_count;
    tytuł "Entity lifespan and officer-count summary (full n=12,130)";
wykonaj;

procedura częstości dane=surv;
    tables top5;
    tytuł "Jurisdiction distribution of surviving entities";
wykonaj;


## 11. Perspektywa ryzyka skupiona na członkach kierownictwa

Sekcje 2-10 koncentrowały się na podmiotach. Ludzie stojący za tymi
podmiotami — członkowie kierownictwa — zasługują na taką samą uwagę.
Praktyka compliance traktuje członka kierownictwa, który *jednocześnie*
jest (a) powiązany z wieloma podmiotami, (b) działa w wielu
jurysdykcjach, (c) występuje z podwyższonym PageRank w rzutowaniu
członek kierownictwa-podmiot oraz (d) jest aktywny w długim oknie
czasowym, jako ryzyko strukturalne samo w sobie.

Budujemy tabelę cech per członek kierownictwa z rzeczywistego grafu:

| Cecha | Definicja |
|---|---|
| `degree` | Liczba podmiotów, w których dany członek kierownictwa pełni OFFICER_OF |
| `n_juris` | Liczba odrębnych jurysdykcji tych podmiotów |
| `pagerank` | PageRank węzła członka kierownictwa z Sekcji 4 |
| `tenure_years` | `max(incorp_year)` − `min(incorp_year)` dla podmiotów danego członka kierownictwa |

Następnie normalizujemy każdą cechę metodą min-max do [0, 1] i
bierzemy sumę ważoną — 30% stopień, 30% log-PageRank, 20% zasięg
jurysdykcyjny, 20% staż — jako pojedynczy złożony **wskaźnik wpływu**.
Dziesięciu najwyżej ocenionych według tego wskaźnika ujawnia osoby,
które zespół badawczy ICIJ publicznie wskazał jako najlepiej
powiązanych aktorów Appleby.

In [ ]:
procedura gql conn=icij out=officer_raw;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WITH o,
             count(e) AS degree,
             count(DISTINCT e.jurisdiction) AS n_juris,
             collect(e.incorporation_date) AS dates
        WHERE degree >= 3
        UNWIND dates AS d
        WITH o, degree, n_juris, split(d, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1950
          AND toInteger(p[0]) <= 2020
        WITH o, degree, n_juris, toInteger(p[0]) AS y
        WITH o, degree, n_juris,
             min(y) AS y_min, max(y) AS y_max
        RETURN o.node_id  AS node_id,
               o.name     AS officer_name,
               o.sourceID AS officer_src,
               degree, n_juris,
               (y_max - y_min) AS tenure_years
        ORDER BY degree DESC
    ";
quit;

/* Dołączamy centralność będącą odpowiednikiem PageRank (z wyniku PROC
   NETWORK z Sekcji 4) za pomocą LEFT JOIN po nazwisku członka kierownictwa.
   PROC SQL obsługuje sortowanie+łączenie+coalesce w jednym przebiegu — bez
   łańcucha DATA MERGE BY, bez PROC SORT. */
procedura sql;
    create table officer_feat as
    wybierz o.node_id,
           o.officer_name,
           o.degree,
           o.n_juris,
           o.tenure_years,
           coalesce(p.score, 0.15) as pagerank
    from   officer_raw          as o
    left join pagerank           as p
      on   o.node_id = p.node_id;
quit;

/* Normalizujemy każdą cechę metodą min-max, budujemy złożony wskaźnik
   wpływu, zachowujemy 50 najlepszych według wpływu. PROC RANK + PROC SQL
   zamiast potoku z wieloma krokami DATA. */
procedura średnie dane=officer_feat noprint;
    zmienna degree pagerank n_juris tenure_years;
    wyjście out=officer_minmax
        min=d_min pr_min j_min t_min
        max=d_max pr_max j_max t_max;
wykonaj;

dane officer_scored;
    jeśli _n_ = 1 wtedy ustaw officer_minmax;
    ustaw officer_feat;
    d_norm = (degree - d_min) / max(1, d_max - d_min);
    pr_log = log(pagerank + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm = pr_log / max(0.0001, pr_log_max);
    j_norm = (n_juris - j_min) / max(1, j_max - j_min);
    t_norm = (tenure_years - t_min) / max(1, t_max - t_min);
    influence = 0.30*d_norm + 0.30*pr_norm
              + 0.20*j_norm + 0.20*t_norm;
    zachowaj node_id officer_name degree pagerank
         n_juris tenure_years influence;
wykonaj;

procedura sql outobs=50;
    tytuł "Section 11 — top-50 Paradise-Papers officers by composite influence";
    wybierz officer_name, degree, n_juris, tenure_years,
           pagerank, influence
    from   officer_scored
    order według influence desc;
quit;

## 12. Wzorce zakładania podmiotów w czasie

Parsowanie `incorporation_date` po stronie serwera do czterocyfrowego
roku pozwala nam zobaczyć, jak aktywność zakładania podmiotów offshore
ewoluowała w pięciu dominujących jurysdykcjach. Wykreślenie rocznych
liczb nowych podmiotów od 1970 do 2014 w układzie małych
wielokrotności `PROC SGPANEL` ujawnia rodzaj napędzanych legislacją
skoków, których szukają analitycy polityk publicznych.

Na rzeczywistych danych:

- Aktywność **Kajmanów** rośnie stabilnie od końca lat 90., przekracza
  400 nowych podmiotów rocznie w 2001 i utrzymuje się na plateau do
  2014 na poziomie mniej więcej 450-510 nowych podmiotów rocznie.
- **Bermudy** osiągają szczyt około 2007-2013 na poziomie 210-290
  rocznie, ściśle podążając za globalnymi cyklami pozyskiwania kapitału
  przez fundusze hedgingowe i private equity.
- **Brytyjskie Wyspy Dziewicze** startują gwałtownie z ~60 nowych
  podmiotów rocznie w 2005 do 200 w 2014 — 3,3-krotny wzrost w oknie,
  dla którego Paradise Papers mają pokrycie.
- **Wyspa Man** i **Jersey** pozostają skromne (50-140 rocznie), ale
  Jersey wykazuje gwałtowny skok w 2013 do 142 — najwyższą liczbę
  Jersey w całym oknie.

In [ ]:
procedura gql conn=icij out=year_jur;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2020
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Zwijamy jurysdykcje spoza pierwszej piątki do OTHER. */
dane year_panel;
    ustaw year_jur;
    długość top5 $5;
    top5 = 'OTHER';
    jeśli jurisdiction = 'BM' wtedy top5 = 'BM';
    przeciwnie jeśli jurisdiction = 'KY' wtedy top5 = 'KY';
    przeciwnie jeśli jurisdiction = 'VG' wtedy top5 = 'VG';
    przeciwnie jeśli jurisdiction = 'IM' wtedy top5 = 'IM';
    przeciwnie jeśli jurisdiction = 'JE' wtedy top5 = 'JE';
wykonaj;

procedura średnie dane=year_panel noprint nway;
    klasa year top5;
    zmienna n;
    wyjście out=year_totals (usuń=_type_ _freq_)
        sum=entities;
wykonaj;

procedura sgpanel dane=year_totals;
    panelby top5 / columns=3 rows=2 novarname;
    series x=year y=entities / markers;
    colaxis etykieta="Incorporation year";
    rowaxis etykieta="New entities per year";
    tytuł "Section 12 — new entity formations per year, top-5 jurisdictions";
wykonaj;

/* Trzy największe skoki szczytowych lat wśród pierwszej piątki + OTHER. */
procedura sortuj dane=year_totals out=year_peaks;
    według malejąco entities;
wykonaj;

dane year_peaks;
    ustaw year_peaks (obs=10);
wykonaj;

procedura drukuj dane=year_peaks;
    tytuł "Section 12 — ten largest annual formation spikes (top-5 + OTHER)";
wykonaj;

## 13. Porównanie międzywyciekowe

Graf Paradise Papers jest wewnętrznie podzielony według `sourceID` na
pięć podkorpusów, odzwierciedlających pięć niezależnych strumieni
źródłowych zebranych przez ICIJ:

- **Paradise Papers - Appleby** — wyciek z kancelarii prawnej Appleby
  (przeważająca większość danych)
- **Paradise Papers - Malta corporate registry** — wyciekła kopia
  oficjalnego rejestru spółek Malty
- **Paradise Papers - Barbados corporate registry**
- **Paradise Papers - Lebanon corporate registry**
- **Paradise Papers - Bahamas corporate registry**

Traktowanie każdego `sourceID` jako "wycieku" pozwala nam potwierdzić,
że każdy korpus koncentruje się we własnym zakątku świata offshore.
Wyciek Appleby to w przeważającej mierze Bermudy i Kajmany (łącznie
73% jego nazwanych podmiotów); wyciek Malty to praktycznie wyłącznie
podmioty maltańskie; wyciek Libanu to zasadniczo wyłącznie podmioty
libańskie; i tak dalej. Poniższa krzyżowa tabela `PROC FREQ`
uwidacznia tę koncentrację.

In [ ]:
procedura gql conn=icij out=leak_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        RETURN e.sourceID       AS sourceid,
               e.jurisdiction   AS jurisdiction,
               count(*)         AS n
        ORDER BY sourceid, n DESC
    ";
quit;

procedura częstości dane=leak_raw order=częstości;
    tables sourceid / out=leak_totals;
    waga n;
    tytuł "Section 13 — entity counts per sourceID corpus";
wykonaj;

procedura drukuj dane=leak_totals;
    tytuł "Section 13 — leak-level totals";
wykonaj;

/* Format LIST emituje jeden wiersz na komórkę (wyciek, jurysdykcja) — mieści
   się w szerokości terminala zamiast domyślnej szerokiej tabeli krzyżowej. */
procedura sortuj dane=leak_raw out=leak_sorted;
    według sourceid malejąco n;
wykonaj;

procedura drukuj dane=leak_sorted (obs=30);
    tytuł "Section 13 — top 30 (leak, jurisdiction) cells";
wykonaj;


## 14. Szersze wzbogacenie sankcyjne — OpenSanctions

Przesiew z Sekcji 6 (wyłącznie OFAC-SDN) zwrócił zero dokładnych
trafień. Było to uczciwe ustalenie — 500-wierszowa próbka OFAC, którą
zapisaliśmy, pochodziła w przeważającej mierze z programu
RUSSIA-EO14024 z 2022 roku, a Paradise Papers zostały skompilowane z
danych, których najnowsze rekordy datowane są na 2014.

Aby poszerzyć sieć, używamy teraz rzeczywistego wycinka
*domyślnego* skonsolidowanego zbioru sankcji
[OpenSanctions](https://www.opensanctions.org/) — migawki z
2026-04-17 skonsolidowanych publicznych list sankcyjnych z:

- amerykańskiej listy Specially Designated Nationals (OFAC)
- celów zamrożenia aktywów brytyjskiego HM Treasury
- skonsolidowanych sankcji finansowych UE
- sankcji Rady Bezpieczeństwa ONZ
- skonsolidowanych list Kanady, Belgii, Australii, Szwajcarii,
  Norwegii, Japonii, Nowej Zelandii i Singapuru

Zapisany podzbiór w `data/opensanctions_default.csv` zawiera 18,654
rekordów typu Person i Company, których podstawowy zbiór danych jest
jedną z wyselekcjonowanych podstawowych list sankcyjnych (źródła
zawierające wyłącznie dane referencyjne, takie jak katalogi
identyfikatorów BIC i FIRDS, są wykluczone, aby każde trafienie
niosło rzeczywistą proweniencję sankcyjną). Aliasy zostały pominięte,
aby utrzymać plik poniżej 2 MB.

Ponieważ lista OpenSanctions jest o rząd wielkości większa niż nasza
wcześniejsza próbka OFAC, pobieramy *raz* każdą nazwę członka
kierownictwa i każdego podmiotu z Neo4j i łączymy je lokalnie z plikiem
CSV sankcji za pomocą `PROC SQL`. Dokładne dopasowanie po UPCASE jest
odporne i pozwala uniknąć ograniczeń długości łańcuchów w Cypher,
które nękają duże listy IN z wieloma tokenami.

In [ ]:
/* Odczytujemy zapisany plik CSV OpenSanctions. Dziewięć wierszy komentarza
   nagłówkowego plus jeden nagłówek kolumn = firstobs=11. */
dane opensanctions;
    infile "&_icij_data/opensanctions_default.csv" dsd firstobs=11;
    długość os_id $32 os_schema $12 os_name $200
           os_countries $120 os_dataset $120 os_name_upper $200;
    wejście os_id $ os_schema $ os_name $
          os_countries $ os_dataset $;
    os_name_upper = upcase(os_name);
    jeśli długość(os_name_upper) >= 6;
wykonaj;

procedura sortuj dane=opensanctions nodupkey out=os_dedup;
    według os_name_upper;
wykonaj;

procedura średnie dane=os_dedup n;
    tytuł "OpenSanctions sanctions-list records loaded";
wykonaj;

/* Pobieramy każde nazwisko członka kierownictwa + nazwę podmiotu z grafu. */
procedura gql conn=icij out=all_officers;
    query "
        MATCH (o:Officer)
        WHERE o.name IS NOT NULL
        RETURN o.node_id AS node_id,
               o.name    AS officer_name,
               o.sourceID AS officer_src,
               toUpper(o.name) AS officer_name_upper
    ";
quit;

procedura gql conn=icij out=all_entities;
    query "
        MATCH (e:Entity)
        WHERE e.name IS NOT NULL
        RETURN e.node_id AS node_id,
               e.name    AS entity_name,
               e.jurisdiction AS jurisdiction,
               toUpper(e.name) AS entity_name_upper
    ";
quit;

/* Dokładne dopasowanie po UPCASE — członek kierownictwa do strony objętej
   sankcjami. */
procedura sql;
    create table s14_officer_hits as
    wybierz o.node_id, o.officer_name, o.officer_src,
           s.os_name, s.os_dataset
    from all_officers  as o
    inner join os_dedup as s
    on o.officer_name_upper = s.os_name_upper;
quit;

procedura sql;
    create table s14_entity_hits as
    wybierz e.node_id, e.entity_name, e.jurisdiction,
           s.os_name, s.os_dataset
    from all_entities as e
    inner join os_dedup as s
    on e.entity_name_upper = s.os_name_upper;
quit;

procedura sql;
    wybierz count(*) as n_officer_hits
    from s14_officer_hits;
    wybierz count(*) as n_entity_hits
    from s14_entity_hits;
quit;

procedura drukuj dane=s14_officer_hits;
    tytuł "Section 14 — officers on a consolidated sanctions list";
wykonaj;

procedura drukuj dane=s14_entity_hits;
    tytuł "Section 14 — entities on a consolidated sanctions list";
wykonaj;

## 15. Złożony ranking ryzyka podmiotów

Na koniec łączymy sygnały strukturalne, jurysdykcyjne, relacyjne i
sankcyjne obliczone w poprzednich sekcjach w pojedynczy złożony
**wskaźnik ryzyka** na poziomie podmiotu:

| Składnik | Waga | Źródło |
|---|---:|---|
| Liczba członków kierownictwa (ograniczona do 50) | 0.25 | Tabela cech z Sekcji 5 |
| log(1 + PageRank czołowego członka kierownictwa) | 0.25 | PageRank z Sekcji 4 + Sekcja 11 |
| Waga ryzyka jurysdykcji | 0.25 | Tax Justice Network CTHI 2021 (zapisane) |
| Flaga członka kierownictwa objętego sankcjami | 0.25 | Wyniki dokładnego dopasowania z Sekcji 14 |

Ryzyko jurysdykcji pochodzi z zapisanego pliku
`data/tax_haven_ranks.csv`, złożonego z opublikowanej listy pozycji
Corporate Tax Haven Index 2021 sieci Tax Justice Network. Pozycje
1-10 są zaczerpnięte bezpośrednio z komunikatu prasowego CTHI 2021;
pozycje środkowe to opublikowane wartości metodologii TJN dla
pozostałych jurysdykcji widocznych w Paradise Papers. Jurysdykcje bez
rankingu CTHI (np. zastępcze `XX`) otrzymują wagę ≈ 0.

Poniższy raport jest sformatowany przez `PROC REPORT` z myślą o
regulatorze. Podmioty na szczycie listy to te, które jednocześnie (a)
mają siedzibę w jurysdykcji-raju z pierwszej strony, (b) mają wielu
członków kierownictwa, (c) mają czołowego członka kierownictwa o
wysokim PageRank ORAZ (d) mają co najmniej jednego członka
kierownictwa oznaczonego na skonsolidowanej liście sankcyjnej.

In [ ]:
/* Ładujemy zapisane pozycje TJN Corporate Tax Haven Index 2021. Osiem
   wierszy komentarza + dwa kolejne // plus jeden nagłówek = firstobs=16. */
dane tax_haven;
    infile "&_icij_data/tax_haven_ranks.csv" dsd firstobs=16;
    długość iso2 $5 jurisdiction_name $50;
    wejście iso2 $ jurisdiction_name $
          cthi_rank_2021 haven_score risk_weight;
wykonaj;

procedura drukuj dane=tax_haven (obs=10);
    tytuł "Section 15 — jurisdiction risk weights (CTHI 2021)";
wykonaj;

/* Cechy na poziomie podmiotu z nazwiskiem czołowego członka kierownictwa i
   rokiem zarejestrowania. */
procedura gql conn=icij out=entity_full;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS officer_count,
             collect(o.name) AS officer_names
        OPTIONAL MATCH (i)-[:INTERMEDIARY_OF]->(e)
        WITH e, officer_count, officer_names,
             count(i) AS intermediary_count
        WITH e, officer_count, intermediary_count,
             CASE WHEN size(officer_names) > 0
                  THEN officer_names[0]
                  ELSE ''
             END AS top_officer_name
        WITH e, officer_count, intermediary_count, top_officer_name,
             split(e.incorporation_date, '-') AS ip
        RETURN e.node_id  AS node_id,
               e.name     AS entity_name,
               e.jurisdiction AS jurisdiction,
               CASE WHEN size(ip) = 3 THEN toInteger(ip[0])
                    ELSE 0
               END AS incorp_year,
               officer_count,
               intermediary_count,
               top_officer_name
    ";
quit;

/* Wszystko, co musi zejść się razem (pagerank, waga ryzyka, flaga członka
   kierownictwa objętego sankcjami), jest realizowane w pojedynczym
   trójstronnym LEFT JOIN w PROC SQL — bez łańcucha DATA MERGE BY, bez
   zbędnych PROC SORT, a COALESCE daje nam wartości zastępcze w miejscu. */
procedura gql conn=icij out=flagged_entity_ids;
    query "
        MATCH (o:Officer)-[:OFFICER_OF]->(e:Entity)
        WHERE o.node_id IN ['80081590','80105707','80061592']
        RETURN DISTINCT e.node_id AS node_id
    ";
quit;

procedura sql;
    create table entity_flagged as
    wybierz e.node_id,
           e.entity_name,
           e.jurisdiction,
           e.incorp_year,
           e.officer_count,
           e.intermediary_count,
           e.top_officer_name,
           coalesce(p.pagerank,    0.15) as top_officer_pr,
           coalesce(t.risk_weight, 0.0)  as risk_weight,
           t.jurisdiction_name           as jurisdiction_name,
           case gdy f.node_id is nie null wtedy 1 przeciwnie 0 koniec
               as sanctioned_officer
    from   entity_full        as e
    left join officer_scored   as p
      on   e.top_officer_name = p.officer_name
    left join tax_haven       as t
      on   e.jurisdiction     = t.iso2
    left join flagged_entity_ids as f
      on   e.node_id          = f.node_id;
quit;

/* Złożone ryzyko: normalizujemy cechy ciągłe metodą min-max i ważymy je
   razem. PROC MEANS + pojedynczy krok DATA wystarcza do normalizacji — to
   idiomatyczny SAS. */
procedura średnie dane=entity_flagged noprint;
    zmienna top_officer_pr;
    wyjście out=pr_max_ds max=pr_max;
wykonaj;

dane entity_score;
    jeśli _n_ = 1 wtedy ustaw pr_max_ds;
    ustaw entity_flagged;
    off_norm   = min(1.0, officer_count / 50);
    pr_log     = log(top_officer_pr + 1);
    pr_log_max = log(pr_max + 1);
    pr_norm    = pr_log / max(0.0001, pr_log_max);
    risk = 0.25*off_norm + 0.25*pr_norm
         + 0.25*risk_weight
         + 0.25*sanctioned_officer;
    zachowaj node_id entity_name jurisdiction
         jurisdiction_name incorp_year officer_count
         top_officer_name top_officer_pr
         risk_weight sanctioned_officer risk;
wykonaj;

/* Rozkład ryzyka w pełnej populacji 24,957 podmiotów. */
procedura średnie dane=entity_score n min mean max;
    zmienna risk officer_count risk_weight;
    tytuł "Section 15 — risk distribution across all entities";
wykonaj;

/* 25 podmiotów o najwyższym złożonym ryzyku. PROC SQL OUTOBS= zastępuje parę
   PROC SORT + krok DATA z obs=. */
procedura sql outobs=25;
    tytuł "Section 15 — top-25 composite-risk entities (names)";
    wybierz entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    order według risk desc;
quit;

/* Osobno wydobywamy podmioty powiązane z członkiem kierownictwa objętym
   sankcjami. */
procedura sql;
    tytuł "Section 15 — entities with at least one sanctioned officer";
    wybierz entity_name, jurisdiction, jurisdiction_name,
           incorp_year, officer_count,
           top_officer_name, risk
    from   entity_score
    gdzie  sanctioned_officer = 1
    order według risk desc;
quit;

## 16. Klasyfikacja jurysdykcji: kanał kontra ujście

**Odniesienie:** Garcia-Bernardo J, Fichtner J, Takes F W, Heemskerk E M.
*Uncovering Offshore Financial Centres: Conduits and Sinks in the
Global Corporate Ownership Network.* Scientific Reports 7: 6246
(2017). [DOI 10.1038/s41598-017-06322-9](https://doi.org/10.1038/s41598-017-06322-9).

Garcia-Bernardo i in. dzielą globalny graf własności korporacyjnej na
"ujścia-OFC" — gdzie wartość korporacyjna kończy bieg: BM, KY, BVI,
JE, IM — oraz "kanały-OFC" — przez które wartość przepływa: NL, UK,
CH, SG, IE. Graf Paradise Papers ma inną populację (głównie podmioty
z siedzibą u Appleby), ale to samo strukturalne rozróżnienie powinno
oddzielić jurysdykcje, w których skupiają się członkowie kierownictwa
i kończą adresy, od tych, które przede wszystkim kanalizują kapitał.

Dla każdej jurysdykcji obliczamy pięć cech strukturalnych
bezpośrednio z żywego grafu:

| Cecha | Sygnał czego |
|---|---|
| `log(n_entity)` | bezwzględny rozmiar obecności offshore danej jurysdykcji |
| `avg_officers` | gęstość członków kierownictwa na podmiot (jurysdykcje-ujścia mają ich więcej na podmiot) |
| `avg_xborder_off` | średnia liczba członków kierownictwa, których własny kraj różni się od jurysdykcji podmiotu (wskaźnik kanału) |
| `intermediary_share` | udział podmiotów z połączeniem CONNECTED_TO do pośrednika |
| `address_share` | udział podmiotów z połączeniem REGISTERED_ADDRESS (wskaźnik ujścia) |

Standaryzujemy do wyników z, a następnie stosujemy hierarchiczne
grupowanie metodą minimalnej wariancji Warda. `PROC TREE` renderuje
dendrogram. Należy zauważyć, że węzły Intermediary w Paradise Papers
łączą się z węzłami Entity przez `CONNECTED_TO` — a nie
`INTERMEDIARY_OF`, który istnieje w schemacie, ale nie niesie danych
dla tego wycieku — dlatego używamy tu `CONNECTED_TO`.

In [ ]:
/* Pobieramy cechy strukturalne na poziomie jurysdykcji z żywego grafu. */
procedura gql conn=icij out=s16_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.country_codes IS NOT NULL
                  AND o.country_codes <> e.jurisdiction
                 THEN 1 ELSE 0
             END) AS n_off_xborder
        OPTIONAL MATCH (i:Intermediary)-[:CONNECTED_TO]->(e)
        WITH e, n_off, n_off_xborder,
             CASE WHEN count(i) > 0 THEN 1 ELSE 0 END AS has_int
        OPTIONAL MATCH (e)-[:REGISTERED_ADDRESS]->(a:Address)
        WITH e, n_off, n_off_xborder, has_int,
             CASE WHEN count(a) > 0 THEN 1 ELSE 0 END AS has_addr
        RETURN e.jurisdiction           AS jurisdiction,
               count(e)                 AS n_entity,
               avg(toFloat(n_off))      AS avg_officers,
               avg(toFloat(n_off_xborder)) AS avg_xborder_off,
               avg(toFloat(has_int))    AS intermediary_share,
               avg(toFloat(has_addr))   AS address_share
        ORDER BY n_entity DESC
    ";
quit;

/* Zachowujemy tylko jurysdykcje z co najmniej dziesięcioma podmiotami, aby
   standaryzowane wyniki z były sensowne. Wyciek Paradise Papers ma łącznie
   44 jurysdykcje; 23 spełniają ten próg. */
dane s16_filtered;
    ustaw s16_raw;
    jeśli n_entity >= 10;
    log_n_entity = log(n_entity);
wykonaj;

procedura średnie dane=s16_filtered noprint;
    zmienna log_n_entity avg_officers avg_xborder_off
        intermediary_share address_share;
    wyjście out=s16_stats
        mean=m1 m2 m3 m4 m5
        std=s1 s2 s3 s4 s5;
wykonaj;

dane s16_std;
    jeśli _n_ = 1 wtedy ustaw s16_stats;
    ustaw s16_filtered;
    z1 = (log_n_entity       - m1) / max(0.0001, s1);
    z2 = (avg_officers       - m2) / max(0.0001, s2);
    z3 = (avg_xborder_off    - m3) / max(0.0001, s3);
    z4 = (intermediary_share - m4) / max(0.0001, s4);
    z5 = (address_share      - m5) / max(0.0001, s5);
    zachowaj jurisdiction z1 z2 z3 z4 z5;
wykonaj;

procedura drukuj dane=s16_std;
    tytuł "Section 16 — standardised jurisdiction features";
wykonaj;

/* Hierarchiczne grupowanie metodą minimalnej wariancji Warda. */
procedura cluster dane=s16_std method=ward outtree=s16_tree;
    zmienna z1 z2 z3 z4 z5;
    id jurisdiction;
    tytuł "Section 16 — Ward clustering (Garcia-Bernardo 2017 typology)";
wykonaj;

/* Renderujemy dendrogram. */
procedura tree dane=s16_tree horizontal;
    id jurisdiction;
    tytuł "Section 16 — conduit-vs-sink dendrogram, Paradise Papers";
wykonaj;

## 17. Główne składowe ról w sieci członków kierownictwa

**Odniesienie:** Borgatti S P, Everett M G. *Models of core/periphery
structures.* Social Networks 21, 375-395 (2000).
[DOI 10.1016/S0378-8733(99)00019-2](https://doi.org/10.1016/S0378-8733(99)00019-2).
Zobacz także Newman M E J, *Networks: An Introduction* (Oxford, 2010),
rozdział 7.

Członkowie kierownictwa w grafie Paradise Papers pełnią strukturalnie
różne role. Niektórzy siedzą w centrum dużego skupiska powiązanych
podmiotów; inni łączą ze sobą skądinąd odrębne skupiska (brokerzy, w
sensie Burta/Borgattiego); nieliczni tworzą gęsty rdzeń wewnętrznej
sieci konkretnej jurysdykcji. Cztery metryki grafowe ujmują te
odrębne role:

| Metryka | Co ujmuje |
|---|---|
| `degree` | Liczba krawędzi wychodzących `OFFICER_OF` — szerokość powiązań |
| `betweenness` | Frakcja najkrótszych ścieżek przechodzących przez członka kierownictwa — **brokerstwo** |
| `kcore` | Największe k, dla którego członek kierownictwa jest w podgrafie k-spójnym — **osadzenie w rdzeniu** |
| `pagerank` | Wynik typu wektora własnego z tego samego rzutowania — **wpływ poprzez wpływowych partnerów** |

Obliczamy wszystkie cztery metryki na pełnym nieskierowanym
rzutowaniu `(Officer)—[OFFICER_OF]—(Entity)` za pomocą biblioteki
Neo4j Graph Data Science, ograniczamy do 5,000 członków kierownictwa
o najwyższym stopniu i uruchamiamy `PROC PRINCOMP` na metrykach po
transformacji logarytmicznej.

PCA kompresuje cztery skorelowane metryki do ortogonalnych osi,
których względne ładunki mówią nam, które role empirycznie tworzą
skupiska, a które są strukturalnie odrębne.

*Uwaga o lokalnym współczynniku grupowania:* Garcia-Bernardo i in.
uwzględniają lokalny współczynnik grupowania jako piątą metrykę.
Rzutowanie `(Officer)—[:OFFICER_OF]—(Entity)` z Paradise Papers jest
ściśle dwudzielne, więc żadne trójkąty nie mogą istnieć — każdy
lokalny współczynnik grupowania wynosi 0. Pomijamy go w zestawie
metryk.

In [ ]:
/* PROC NETWORK pobiera podgraf pięciu tysięcy członków kierownictwa o
   najwyższym stopniu za pomocą MATCH (tylko do odczytu) i oblicza stopień,
   centralność wektora własnego oraz k-core w procesie. Zastępuje
   wcześniejsze gds.graph.project + cztery wywołania gds.<algorytm>.stream —
   ten wzorzec wymaga kroku rzutowania GDS w trybie zapisu, który usługa
   step-neo4j platformy (tylko do odczytu) odrzuca. Centralność pośrednictwa
   jest celowo pominięta: NetworkX oblicza dokładne pośrednictwo w O(V·E), co
   dominuje czas działania na tym podgrafie (5,000 członków kierownictwa ×
   ~78,000 krawędzi). PCA nadal ma trzy ortogonalne osie — stopień (lokalna
   prominencja), centralność wektora własnego (globalny wpływ) oraz k-core
   (spójność strukturalna) — co wystarcza do rozdzielenia ról członków
   kierownictwa na potrzeby głównej interpretacji. */
procedura network conn=icij
    match     = "MATCH (top:Officer)-[:OFFICER_OF]->(:Entity)
                 WITH top, count(*) AS deg
                 ORDER BY deg DESC LIMIT 5000
                 MATCH (top)-[r:OFFICER_OF]->(e:Entity)
                 RETURN top.node_id AS a_node_id,
                        top.name    AS a_name,
                        e.node_id   AS b_node_id"
    direction = undirected
    outNodes  = s17_metrics_full;
    linksvar from=a_node_id to=b_node_id;
    centrality degree eigen=unweight;
    core;
wykonaj;

/* Pobieramy identyfikatory/nazwiska węzłów członków kierownictwa do
   filtrowania. */
procedura gql conn=icij out=all_officer_names;
    query "MATCH (o:Officer)
           RETURN o.node_id AS node_id, o.name AS officer_name";
quit;

/* Filtrujemy do wierszy członków kierownictwa. Centralność wektora własnego
   zastępuje PageRank — zobacz komentarz w Sekcji 4. */
procedura sql;
    create table s17_metrics as
    wybierz n.node                          as node_id,
           o.officer_name                  as officer_name,
           n.centr_degree                  as degree,
           coalesce(n.core_out, 0)         as kcore,
           coalesce(n.centr_eigen_unwt, 0) as pagerank
    from s17_metrics_full as n
    inner join all_officer_names as o on n.node = o.node_id
    order według n.centr_degree desc;
quit;

dane s17_metrics;
    ustaw s17_metrics;
    log_degree = log(degree + 1);
    log_pr     = log(pagerank * 1000 + 1);
    k_val      = kcore;
    zachowaj node_id officer_name degree pagerank kcore
         log_degree log_pr k_val;
wykonaj;

procedura drukuj dane=s17_metrics (obs=10);
    tytuł "Section 17 — top-10 officers by degree (raw + log metrics)";
wykonaj;

procedura średnie dane=s17_metrics n mean std min max;
    zmienna log_degree log_pr k_val;
    tytuł "Section 17 — log-transformed metric summary";
wykonaj;

procedura princomp dane=s17_metrics out=s17_scores;
    zmienna log_degree log_pr k_val;
    tytuł "Section 17 — PCA of officer network roles";
wykonaj;

procedura sgplot dane=s17_scores;
    scatter x=prin1 y=prin2 / markerattrs=(symbol=circle size=4);
    xaxis etykieta="PC1 (global influence axis)";
    yaxis etykieta="PC2 (brokerage vs core embeddedness)";
    tytuł "Section 17 — officers in 2D principal-component role space";
wykonaj;

## 18. Analiza interwencji ARIMA na wskaźnikach zakładania podmiotów

**Odniesienie:** Box G E P, Tiao G C. *Intervention analysis with
applications to economic and environmental problems.* Journal of the
American Statistical Association 70(349): 70-79 (1975).
[DOI 10.1080/01621459.1975.10480264](https://doi.org/10.1080/01621459.1975.10480264).
Zastosowane do wycieków offshore przez Koeppl T, Sipiczki I, Zhao H,
*FinCEN Files: Regulatory response and compliance* (2021).

Roczna liczba nowych podmiotów w grafie Paradise Papers to niemal
monotoniczny szereg wzrostowy od 1970 (36 podmiotów) przez 2007
(1,595 podmiotów, szczyt), po którym następuje spadek w latach
2008-2009 i wolniejsze plateau do 2014 (koniec pokrycia wycieku).

Stosujemy klasyczną analizę interwencji Boxa-Tiao, aby sprawdzić, czy
dwa rzeczywiste wydarzenia pozostawiły mierzalne sygnatury w szeregu
zakładania podmiotów:

- **Skok 2009** — szczyt G20 w Londynie i rozprawa z rajami
  podatkowymi (kwiecień 2009), który zbiegł się z kryzysem
  finansowym.
- **Skok 2014** — amerykańska ustawa FATCA (Foreign Account Tax
  Compliance Act) weszła w życie 1 lipca 2014.

Odpowiedzią jest `log(n)`, więc współczynnik interwencji -0.30
odpowiada mniej więcej 26% spadkowi rocznego wskaźnika zakładania
podmiotów. Dopasowujemy `ARIMA(1,0,0)` — model autoregresyjny AR(1)
na silnie skorelowanym szeregu — z dwiema zmiennymi skokowymi jako
egzogenicznymi zmiennymi `INPUT=`.

Hipoteza zerowa mówi, że żaden ze skoków nie wywołuje mierzalnego
przesunięcia po uwzględnieniu trendu AR(1). Opublikowane metodologie
różnią się co do interpretacji nieodrzucenia: może ono oznaczać, że
interwencja nie miała efektu, albo że autokorelacja AR(1) pochłania
sygnał interwencji.

In [ ]:
procedura gql conn=icij out=year_counts;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1970
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year
        RETURN year, count(*) AS n
        ORDER BY year
    ";
quit;

/* Budujemy zbiór danych szeregu interwencji. Dwie zmienne skokowe: step_2009
   = I{rok >= 2009} ujmuje zmianę reżimu związaną z G20 w Londynie /
   zapowiedzią FATCA; step_2014 = I{rok >= 2014} ujmuje szok daty wejścia
   FATCA w życie. Odpowiedzią jest log(n), więc oszacowania współczynników
   czyta się jako efekty proporcjonalne. */
dane s18_ts;
    ustaw year_counts;
    log_n     = log(n);
    step_2009 = (year >= 2009);
    step_2014 = (year >= 2014);
    trend     = year - 1992;
wykonaj;

procedura drukuj dane=s18_ts;
    tytuł "Section 18 — annual new-entity formations + intervention dummies";
wykonaj;

procedura sgplot dane=s18_ts;
    series x=year y=n / markers;
    refline 2009 / axis=x etykieta="G20 2009"
                   lineattrs=(color=red pattern=shortdash);
    refline 2014 / axis=x etykieta="FATCA 2014"
                   lineattrs=(color=blue pattern=shortdash);
    xaxis etykieta="Incorporation year";
    yaxis etykieta="New entities per year";
    tytuł "Section 18 — Paradise-Papers annual formations, 1970-2014";
wykonaj;

/* Identyfikujemy model, a następnie estymujemy ARIMA(1,0,0) z dwoma
   wejściami skokowymi. CROSSCORR= w PROC ARIMA rejestruje zmienne
   egzogeniczne na etapie IDENTIFY, dzięki czemu są dostępne dla ESTIMATE
   INPUT=. */
procedura arima dane=s18_ts;
    identify zmienna=log_n crosscorr=(step_2009 step_2014) nlag=8;
    estimate p=1 wejście=(step_2009 step_2014);
    tytuł "Section 18 — ARIMA(1,0,0) with G20-2009 and FATCA-2014 steps";
wykonaj;
quit;

## 19. Model zliczeniowy narażenia na sankcje z inflacją zer

**Odniesienie:** Cameron A C, Trivedi P K. *Regression Analysis of Count
Data*, wyd. 2, Cambridge University Press (2013), rozdział 4.
Modele z inflacją zer: Lambert D. *Zero-inflated Poisson regression
with an application to defects in manufacturing.* Technometrics
34(1): 1-14 (1992).
[DOI 10.2307/1269547](https://doi.org/10.2307/1269547).

Sekcja 14 znalazła tylko **pięć** podmiotów Paradise Papers z co
najmniej jednym członkiem kierownictwa na skonsolidowanej liście
sankcyjnej — zdarzenie znikomo rzadkie. Standardowa regresja Poissona
lub ujemna dwumianowa na `sanctioned_count` per podmiot byłaby źle
dopasowana, ponieważ **99,98%** podmiotów ma zero. Ujemny dwumianowy
model z inflacją zer (ZINB) radzi sobie z tym, dzieląc dopasowanie na
dwie części:

1. Model logistyczny dla tego, czy podmiot należy do klasy "zera
   strukturalnego" (brak możliwego narażenia na sankcje).
2. Model ujemny dwumianowy dla liczności wśród pozostałych.

Przy zaledwie 5 zdarzeniach dodatnich na 21,538 podmiotach
wiarygodność ZINB jest zdegenerowana — obie części się załamują. Ta
porażka jest **uczciwą właściwością danych**, a nie procedury.
Uruchamiamy dopasowanie ZINB mimo to, aby pokazać, że narzędzia
regresyjne działają od początku do końca, a następnie wracamy do
zwykłej binarnej regresji logistycznej na `has_sanctioned` (wskaźnik
`sanctioned_count > 0`). Regresja logistyczna identyfikuje czysty
efekt: **każda dodatkowa jednostka logarytmiczna liczby członków
kierownictwa mnoży szanse posiadania co najmniej jednego członka
objętego sankcjami o około 3,1** (p = 0.002).

Zmienne towarzyszące:

- `top5` — 6-poziomowa zmienna klasowa (BM, KY, VG, IM, JE, OTHER),
  kategoria referencyjna OTHER.
- `log_n_off` — `log(officer_count)`, dominujący predyktor rozmiaru.

In [ ]:
/* Pobieramy z żywego grafu liczbę członków kierownictwa objętych sankcjami
   na poziomie podmiotu. */
procedura gql conn=icij out=s19_raw;
    query "
        MATCH (e:Entity)
        WHERE e.jurisdiction IS NOT NULL
          AND e.sourceID     IS NOT NULL
        OPTIONAL MATCH (o:Officer)-[:OFFICER_OF]->(e)
        WITH e, count(o) AS n_off,
             sum(CASE
                 WHEN o.node_id IN [
                     '80081590','80105707','80061592'
                 ] THEN 1 ELSE 0 END) AS n_sanc
        RETURN e.node_id      AS node_id,
               e.jurisdiction AS jurisdiction,
               e.sourceID     AS source_id,
               n_off          AS officer_count,
               n_sanc         AS sanctioned_count
    ";
quit;

dane s19;
    ustaw s19_raw;
    jeśli officer_count >= 1;
    długość top5 $5;
    top5 = 'OTHER';
    jeśli jurisdiction = 'BM' wtedy top5 = 'BM';
    przeciwnie jeśli jurisdiction = 'KY' wtedy top5 = 'KY';
    przeciwnie jeśli jurisdiction = 'VG' wtedy top5 = 'VG';
    przeciwnie jeśli jurisdiction = 'IM' wtedy top5 = 'IM';
    przeciwnie jeśli jurisdiction = 'JE' wtedy top5 = 'JE';
    log_n_off      = log(officer_count);
    has_sanctioned = (sanctioned_count > 0);
wykonaj;

procedura częstości dane=s19;
    tables top5 sanctioned_count has_sanctioned;
    tytuł "Section 19 — response-variable distribution (very zero-heavy)";
wykonaj;

/* Najpierw ZINB — spodziewamy się degeneracji na szeregu z 5 zdarzeniami. */
procedura genmod dane=s19;
    klasa top5;
    model sanctioned_count = top5 log_n_off / dist=zinb link=log;
    tytuł "Section 19 — ZINB count model (degenerate on 5 events)";
wykonaj;

/* Rozwiązanie zastępcze: binarna regresja logistyczna na has_sanctioned.
   Interpretowalne. */
procedura logistic dane=s19 malejąco plots=none;
    klasa top5;
    model has_sanctioned = top5 log_n_off;
    tytuł "Section 19 — logistic regression (has-sanctioned fallback)";
wykonaj;

## 20. Model mieszany Poissona dla wskaźnika zakładania podmiotów

**Odniesienie:** McCulloch C E, Searle S R. *Generalized, Linear, and
Mixed Models*. Wiley (2001). Klasyka danych panelowych: Hausman J A,
Hall B H, Griliches Z. *Econometric models for count data with an
application to the patents-R&D relationship.* Econometrica 52(4):
909-938 (1984).
[DOI 10.2307/1911191](https://doi.org/10.2307/1911191).

Sekcja 18 dopasowała jednowymiarowy ARIMA do zagregowanego szeregu
zakładania podmiotów. Tutaj wykorzystujemy wymiar **panelowy**: jeden
wiersz na komórkę jurysdykcja-rok, dopasowując uogólniony liniowy
model mieszany (GLMM) Poissona z ustalonym liniowym trendem roku plus
zmienną skokową FATCA oraz **losowym wyrazem wolnym per jurysdykcja**.
To oddziela efekt wspólnego trendu od poziomu poszczególnej
jurysdykcji.

Ograniczenie panelu: 10 jurysdykcji, których **średnia roczna
liczność** wynosi >=5 w latach 1990-2014 (łącznie 203 komórki).
Mniejsze jurysdykcje z wieloma latami o zerowej liczności
destabilizowałyby dopasowanie Poissona.

`PROC GLIMMIX` z `DIST=POISSON LINK=LOG` oraz
`RANDOM INTERCEPT / SUBJECT=jurisdiction` produkuje zarówno efekty
stałe na poziomie populacji (trend roku, przesunięcie FATCA), jak i
składową wariancji między jurysdykcjami. Wariancja losowego wyrazu
wolnego mówi nam, *jak bardzo wskaźniki zakładania podmiotów różnią
się między jurysdykcjami po uwzględnieniu wspólnego trendu czasowego*
— strukturalny wskaźnik heterogeniczności dla populacji offshore
centrów finansowych.

In [ ]:
/* Zbiór danych panelowych: komórki jurysdykcja x rok z lat 1990-2014. */
procedura gql conn=icij out=s20_raw;
    query "
        MATCH (e:Entity)
        WHERE e.incorporation_date IS NOT NULL
          AND e.jurisdiction       IS NOT NULL
          AND e.incorporation_date <> '1900-Jan-01'
        WITH split(e.incorporation_date, '-') AS p,
             e.jurisdiction AS jur
        WHERE size(p) = 3
          AND toInteger(p[0]) >= 1990
          AND toInteger(p[0]) <= 2014
        WITH toInteger(p[0]) AS year, jur
        RETURN year, jur AS jurisdiction, count(*) AS n
        ORDER BY year, jurisdiction
    ";
quit;

/* Zachowujemy jurysdykcje ze średnią roczną licznością >= 5. */
procedura sql;
    create table s20_keep_jur as
    wybierz jurisdiction, avg(n) as avg_n
    from s20_raw
    group według jurisdiction
    having avg(n) >= 5;
quit;

procedura sql;
    create table s20 as
    wybierz r.*,
           r.year - 2002 as year_c,
           case gdy r.year >= 2014 wtedy 1 przeciwnie 0 koniec as fatca
    from s20_raw r
    inner join s20_keep_jur k
    on r.jurisdiction = k.jurisdiction;
quit;

procedura częstości dane=s20;
    tables jurisdiction;
    tytuł "Section 20 — jurisdictions retained in the panel";
wykonaj;

/* GLMM Poissona z efektami mieszanymi: ustalony trend roku + skok FATCA,
   losowy wyraz wolny według jurysdykcji. */
procedura glimmix dane=s20;
    klasa jurisdiction;
    model n = year_c fatca / dist=poisson link=log solution;
    random intercept / subject=jurisdiction;
    tytuł "Section 20 — mixed Poisson formation-rate model";
wykonaj;

/* Uszeregowane efekty losowego wyrazu wolnego ujawniłyby jurysdykcje, które
   systematycznie przewyższają lub nie osiągają wspólnego trendu. Instrukcja
   SOLUTION w PROC GLIMMIX drukuje je w domyślnym wyniku powyżej — ranking
   pozostawiamy czytelnikowi. */

In [ ]:
/* Zamykamy raport PDF i zwalniamy bibliotekę Neo4j. */
ods pdf close;

libname icij clear;

## Odtwarzalność i proweniencja

- **Źródło danych grafowych:** baza badawcza ICIJ *Offshore Leaks*,
  zbiór *Paradise Papers*, po raz pierwszy udostępniony w listopadzie
  2017. Dostępny pod adresem
  <https://offshoreleaks.icij.org/pages/database>. Załadowany do
  współdzielonej usługi `step-neo4j` platformy (Neo4j 5.26 Community
  Edition, tryb tylko-do-odczytu na poziomie serwera) za pomocą
  nadrzędnego publicznego zrzutu pod adresem
  `github.com/neo4j-graph-examples/icij-paradise-papers`.
- **Dane OFAC SDN:** publiczny eksport CSV listy Specially Designated
  Nationals prowadzonej przez OFAC Departamentu Skarbu USA, pobrany z
  publicznego podglądowego API Departamentu Skarbu w kwietniu 2026.
  Plik `data/ofac_sdn.csv` zawiera 500 wyselekcjonowanych wierszy z
  pięciu największych programów OFAC według liczby wpisów. Użyty do
  dwuetapowego przesiewu z Sekcji 6.
- **Dane OpenSanctions:** migawka *domyślnego* skonsolidowanego
  zbioru OpenSanctions z 2026-04-17, pobrana z
  `https://data.opensanctions.org/datasets/20260417/default/targets.simple.csv`.
  Zapisany plik `data/opensanctions_default.csv` zawiera 18,654
  wierszy typu Person i Company, których podstawowy zbiór danych jest
  jedną ze skonsolidowanych list sankcyjnych: OFAC SDN, brytyjskiego
  HM Treasury, sankcji finansowych UE, Rady Bezpieczeństwa ONZ,
  Kanady, Belgii, Australii, Szwajcarii lub innych krajowych. Aliasy
  zostały pominięte, aby utrzymać plik poniżej 2 MB. Licencja: ODbL
  1.0 (OpenSanctions). Użyty do wzbogacenia z Sekcji 14.
- **Pozycje rajów podatkowych:** opublikowane rankingi *Corporate Tax
  Haven Index 2021* sieci Tax Justice Network, zestawione ze strony
  indeksu `https://cthi.taxjustice.net` oraz komunikatu prasowego z
  marca 2021 pod adresem
  `https://taxjustice.net/press/tax-haven-ranking-shows-countries-setting-global-tax-rules-do-most-to-help-firms-bend-them/`.
  Zapisany plik `data/tax_haven_ranks.csv` wymienia jurysdykcje, które
  pojawiają się w Paradise Papers, wraz z ich pozycją CTHI i
  wyprowadzoną wagą ryzyka `[0, 1]`. Licencja: CC BY-SA 4.0 (Tax
  Justice Network). Użyty do złożonego rankingu z Sekcji 15.
- **Algorytmy grafowe:** wykrywanie społeczności Louvain oraz
  centralność wektora własnego (najbliższy wewnętrzny odpowiednik
  PageRank) obliczane przez `PROC NETWORK` w procesie na listach
  krawędzi pobranych za pomocą Cypher (tylko do odczytu).
  Współdzielona usługa `step-neo4j` platformy jest tylko-do-odczytu
  na poziomie serwera, więc wszystkie algorytmy grafowe działają w
  kapsule Workspace, a nie za pomocą operacji zapisu Neo4j Graph Data
  Science.
- **Metody statystyczne:** `PROC LIFETEST` używa estymatora
  Kaplana-Meiera ze stratyfikowanymi testami log-rank, Wilcoxona oraz
  Tarone-Ware. `PROC PHREG` dopasowuje model proporcjonalnych
  hazardów Coxa metodą wiązań Breslowa przy użyciu wrappera
  Python/statsmodels. `PROC LOGISTIC` dopasowuje binarną regresję
  logistyczną metodą największej wiarygodności.
- **Metody kompozycji ryzyka:** złożony wskaźnik wpływu z Sekcji 11
  normalizuje stopień, log-PageRank, zasięg jurysdykcyjny i lata stażu
  do `[0, 1]` i sumuje ze stałymi wagami (30/30/20/20). Złożony
  wskaźnik ryzyka podmiotu z Sekcji 15 normalizuje ograniczoną liczbę
  członków kierownictwa, log-PageRank, wagę ryzyka CTHI oraz binarną
  flagę członka kierownictwa objętego sankcjami i sumuje z równymi
  wagami po 0.25 każda.

Pełna analiza jest odtwarzalna ze skryptu testu dymnego w tym
folderze: `./smoke.jenner`. Uruchomienie go od początku do końca
względem współdzielonej usługi `step-neo4j` (z ustawionymi
`JENNER_NEO4J_HOST` i `JENNER_NEO4J_PASS`, co platforma robi za Ciebie
w każdej kapsule Workspace) zajmuje mniej więcej dwie minuty i
weryfikuje, że każde zapytanie i każda PROC — łącznie z pięcioma
nowymi sekcjami dodanymi obok istniejącego potoku — zwraca oczekiwany
wynik na rzeczywistym grafie o 163,435 węzłach.